# Stage 3: Feature Engineering Pipeline (Stages 0-3 outputs)

Rebuilds `nvda_features.parquet` and `nvda_modelling_dataset.parquet` after two fixes:
1. Corwin-Schultz spread is now floored at 0 (the value is 0 whenever the estimator's no-arbitrage assumption is violated by the daily OHLCV approximation).
2. Lempel-Ziv complexity now uses a correct LZ-76 implementation, normalised by `n / log₂ n`, so each rolling window produces meaningful variation in [0, ≈1].

Fractional differentiation was fixed in a previous pass and is not modified here.

In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../..'))
from src.features import build_feature_matrix

plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('../../reports/figures', exist_ok=True)

## 1. Load Stage 0–2 inputs

In [ ]:
df       = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
fracdiff = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet')
events   = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet')
labels   = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_labels.parquet')
weights  = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_weights.parquet')

for name, x in [('clean', df), ('fracdiff', fracdiff), ('events', events),
                ('labels', labels), ('weights', weights)]:
    print(f'{name:8s}: {x.shape}')

## 2. Build the modelling dataset

In [ ]:
modelling_dataset = build_feature_matrix(
    df=df, fracdiff=fracdiff, events=events,
    labels=labels, weights=weights,
)
print(f'modelling_dataset shape: {modelling_dataset.shape}')
print(f'columns: {list(modelling_dataset.columns)}')

## 3. Validation — microstructure features

In [ ]:
cs = modelling_dataset['corwin_schultz_spread']
lz = modelling_dataset['lempel_ziv_complexity']
bp = modelling_dataset['bekker_parkinson_vol']
am = modelling_dataset['amihud_illiquidity']
rs = modelling_dataset['roll_spread']

print(f'=== {TICKER} microstructure feature stats ===')
print(f'Corwin-Schultz : min={cs.min():.4f} | max={cs.max():.4f} | mean={cs.mean():.4f} | n_neg={(cs<0).sum()} | n_zero={(cs==0).sum()}')
print(f'Lempel-Ziv     : min={lz.min():.4f} | max={lz.max():.4f} | mean={lz.mean():.4f} | n_unique={lz.nunique()}')
print(f'Bekker-Parkinson: min={bp.min():.4f} | max={bp.max():.4f} | std={bp.std():.4f}')
print(f'Amihud         : min={am.min():.3e} | max={am.max():.3e} | std={am.std():.3e}')
print(f'Roll spread    : min={rs.min():.4f} | max={rs.max():.4f} | n_zero={(rs==0).sum()}')


## 4. NaN / Inf check across every column

In [ ]:
issues = pd.DataFrame({
    'n_nan': modelling_dataset.isna().sum(),
    'n_inf': np.isinf(modelling_dataset.select_dtypes(include='number')).sum(),
})
issues['has_issue'] = (issues['n_nan'] > 0) | (issues['n_inf'] > 0)
if issues['has_issue'].any():
    print('!! columns with NaN or Inf:')
    print(issues[issues['has_issue']])
else:
    print('No NaNs or Infs in any column.')
issues.drop(columns='has_issue')

## 5. Conditional drop of Lempel-Ziv if still degenerate

If `std == 0` after the LZ-76 fix the feature still carries no signal and is removed from both saved parquets, with the reason logged. Otherwise it is kept.

In [ ]:
lz_std = float(modelling_dataset['lempel_ziv_complexity'].std())
lz_unique = int(modelling_dataset['lempel_ziv_complexity'].nunique())

drop_lz = (lz_std == 0) or (lz_unique <= 1)
if drop_lz:
    print(f'Dropping lempel_ziv_complexity: std={lz_std:.6f}, '
          f'n_unique={lz_unique}. Reason: still non-informative '
          'after the LZ-76 fix.')
    modelling_dataset = modelling_dataset.drop(columns=['lempel_ziv_complexity'])
else:
    print(f'Keeping lempel_ziv_complexity: std={lz_std:.6f}, '
          f'n_unique={lz_unique}.')

print()
print(f'final modelling_dataset shape:    {modelling_dataset.shape}')
feature_cols = [c for c in modelling_dataset.columns if c not in ('label','weight','t1')]
print(f'final features ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c}')

## 6. P11 — feature correlation heatmap

In [ ]:
feature_only = modelling_dataset[feature_cols]
corr = feature_only.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        v = corr.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                color='white' if abs(v) > 0.5 else 'black', fontsize=7)
ax.set_title('Feature correlation matrix (event-level)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'P11_feature_correlation_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 7. Save outputs

In [ ]:
feature_cols = [c for c in modelling_dataset.columns if c not in ('label','weight','t1')]
features_only = modelling_dataset[feature_cols]

features_only.to_parquet(f'../../data/processed/per_stock/{TICKER}_ts_features.parquet')
modelling_dataset.to_parquet(f'../../data/processed/per_stock/{TICKER}_modelling_dataset.parquet')

print('Saved:')
print(f'  ../data/processed/nvda_features.parquet           {features_only.shape}')
print(f'  ../data/processed/nvda_modelling_dataset.parquet  {modelling_dataset.shape}')
print(f'  ../reports/figures/P11_feature_correlation_heatmap.png')

## 8. Notes on staleness

Because the feature columns now carry corrected values, the existing Stage 4/5/6 artefacts are stale until those notebooks are re-run:

- `models/model_rf.pkl`, `models/model_xgb.pkl`, `models/model_final.pkl`
- `models/best_params.json`
- `data/processed/cv_results.parquet`
- `data/processed/tuning_log.parquet`
- `data/processed/feature_importance.parquet`

Recommended order to refresh: notebook 06 → 09 → 08.